<a href="https://colab.research.google.com/github/ajose3-ui/individuation-training/blob/main/cool_thing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#  FACE MEMORY TASK — Single Cell Google Colab Script
#  Set FOLDER_PATH to your image folder, then run.
# ============================================================

import os, random, time
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

FOLDER_PATH = "/content/test_data"  # <-- SET YOUR FOLDER PATH HERE

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def load_images(folder):
    paths = [
        os.path.join(folder, f)
        for f in sorted(os.listdir(folder))
        if os.path.splitext(f)[1].lower() in VALID_EXTS
    ]
    if len(paths) < 2:
        raise ValueError(f"Need at least 2 images in '{folder}', found {len(paths)}.")
    return paths

all_image_paths = load_images(FOLDER_PATH)
random.shuffle(all_image_paths)
number_pool = list(range(1, len(all_image_paths) + 1))
random.shuffle(number_pool)
master_pairs = list(zip(all_image_paths, number_pool))
total_faces  = len(master_pairs)

data_log = []

state = {
    "current_dict":   {},
    "pair_index":     0,
    "study_queue":    [],
    "test_queue":     [],
    "step_start":     None,
    "_current_study": None,
}

img_out  = widgets.Output()
ctrl_out = widgets.Output()
display(img_out, ctrl_out)

# ── image helper ──────────────────────────────────────────────
def show_image(path, title=""):
    img = Image.open(path)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(img); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

# ── screens ───────────────────────────────────────────────────
def screen_start():
    with img_out:
        clear_output(wait=True)
    with ctrl_out:
        clear_output(wait=True)
        print("FACE MEMORY TASK")
        print("Study each face and its number, then get tested.")
        print("A new face is added every round.\n")
        btn = widgets.Button(description="Start Task", button_style="primary",
                             layout=widgets.Layout(width="160px"))
        btn.on_click(lambda _: begin_loop())
        display(btn)

def begin_loop():
    to_add = 2 if state["pair_index"] == 0 else 1
    for _ in range(to_add):
        if state["pair_index"] < total_faces:
            p, n = master_pairs[state["pair_index"]]
            state["current_dict"][p] = n
            state["pair_index"] += 1
    study_order = list(state["current_dict"].items())
    random.shuffle(study_order)
    state["study_queue"] = study_order
    next_study_face()

def next_study_face():
    if not state["study_queue"]:
        begin_test()
        return

    path, number = state["study_queue"].pop(0)
    state["step_start"]     = time.time()
    state["_current_study"] = (path, number)

    btn = widgets.Button(description="Next face", button_style="primary",
                         layout=widgets.Layout(width="160px"))

    def on_next(_):
        elapsed = time.time() - state["step_start"]
        p, n = state["_current_study"]
        data_log.append({"path": p, "number": n,
                          "loop_size": len(state["current_dict"]),
                          "phase": "study", "duration_s": elapsed, "correct": None})
        next_study_face()

    btn.on_click(on_next)

    with img_out:
        clear_output(wait=True)
        show_image(path, title=f"Number: {number}")
    with ctrl_out:
        clear_output(wait=True)
        print(f"STUDY  |  {len(state['current_dict'])} faces in set\n")
        display(btn)

def begin_test():
    test_order = list(state["current_dict"].keys())
    random.shuffle(test_order)
    state["test_queue"] = test_order

    btn = widgets.Button(description="Begin Test", button_style="warning",
                         layout=widgets.Layout(width="160px"))
    btn.on_click(lambda _: next_test_face())

    with img_out:
        clear_output(wait=True)
    with ctrl_out:
        clear_output(wait=True)
        print(f"Study phase done.  {len(state['current_dict'])} faces to test.\n")
        display(btn)

def show_test_input(path):
    """Draw the answer input screen."""
    num_input  = widgets.BoundedIntText(min=1, max=total_faces,
                                        description="Number:",
                                        layout=widgets.Layout(width="200px"))
    submit_btn = widgets.Button(description="Submit", button_style="success",
                                layout=widgets.Layout(width="120px"))

    def on_submit(_):
        elapsed     = time.time() - state["step_start"]
        correct_num = state["current_dict"][path]
        is_correct  = (num_input.value == correct_num)
        data_log.append({"path": path, "number": correct_num,
                          "loop_size": len(state["current_dict"]),
                          "phase": "test", "duration_s": elapsed, "correct": is_correct})
        show_test_feedback(is_correct, correct_num)

    submit_btn.on_click(on_submit)

    with ctrl_out:
        clear_output(wait=True)
        print(f"TEST  |  {len(state['current_dict'])} faces in set\n")
        display(widgets.HBox([num_input, submit_btn]))

def show_test_feedback(is_correct, correct_num):
    """Replace controls with feedback + next button."""
    msg = "CORRECT" if is_correct else f"WRONG -- correct answer: {correct_num}"

    next_btn = widgets.Button(description="Next face", button_style="primary",
                              layout=widgets.Layout(width="160px"))
    next_btn.on_click(lambda _: next_test_face())

    with ctrl_out:
        clear_output(wait=True)
        print(f"TEST  |  {len(state['current_dict'])} faces in set\n")
        print(msg + "\n")
        display(next_btn)

def next_test_face():
    if not state["test_queue"]:
        if state["pair_index"] >= total_faces:
            show_stats()
        else:
            begin_loop()
        return

    path = state["test_queue"].pop(0)
    state["step_start"] = time.time()

    with img_out:
        clear_output(wait=True)
        show_image(path, title="What number is this face?")

    show_test_input(path)

# ── stats ─────────────────────────────────────────────────────
def mean(lst):
    return sum(lst) / len(lst) if lst else 0

def show_stats():
    study_logs = [d for d in data_log if d["phase"] == "study"]
    test_logs  = [d for d in data_log if d["phase"] == "test"]
    loop_sizes = sorted(set(d["loop_size"] for d in data_log))

    acc         = {ls: mean([1 if d["correct"] else 0
                    for d in test_logs  if d["loop_size"] == ls]) * 100 for ls in loop_sizes}
    study_times = {ls: mean([d["duration_s"] for d in study_logs if d["loop_size"] == ls])
                   for ls in loop_sizes}
    test_times  = {ls: mean([d["duration_s"] for d in test_logs  if d["loop_size"] == ls])
                   for ls in loop_sizes}

    face_study_time, face_correct_pct = [], []
    for p in {d["path"] for d in data_log}:
        s = [d["duration_s"] for d in study_logs if d["path"] == p]
        c = [d["correct"]    for d in test_logs  if d["path"] == p]
        if s and c:
            face_study_time.append(mean(s))
            face_correct_pct.append(mean([1 if x else 0 for x in c]) * 100)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    xs = loop_sizes

    ax = axes[0, 0]
    ax.bar(xs, [acc[l] for l in xs])
    ax.set_ylim(0, 110); ax.set_xlabel("Loop size"); ax.set_ylabel("Accuracy (%)")
    ax.set_title("Accuracy per Loop"); ax.set_xticks(xs)

    ax = axes[0, 1]
    w, x = 0.35, np.arange(len(xs))
    ax.bar(x - w/2, [study_times[l] for l in xs], width=w, label="Study")
    ax.bar(x + w/2, [test_times[l]  for l in xs], width=w, label="Test")
    ax.set_xlabel("Loop size"); ax.set_ylabel("Avg time (s)")
    ax.set_title("Study vs Test Time per Loop")
    ax.set_xticks(x); ax.set_xticklabels(xs); ax.legend()

    ax = axes[1, 0]
    ct = [d["duration_s"] for d in test_logs if d["correct"]]
    it = [d["duration_s"] for d in test_logs if not d["correct"]]
    db, lb = [], []
    if ct: db.append(ct); lb.append("Correct")
    if it: db.append(it); lb.append("Incorrect")
    if db:
        ax.boxplot(db, patch_artist=True); ax.set_xticklabels(lb)
    ax.set_ylabel("Response time (s)"); ax.set_title("Response Time: Correct vs Incorrect")

    ax = axes[1, 1]
    if face_study_time:
        ax.scatter(face_study_time, face_correct_pct)
    ax.set_xlabel("Avg study time (s)"); ax.set_ylabel("% Correct")
    ax.set_title("Study Time vs Accuracy (per face)")

    plt.suptitle("Face Memory Task -- Summary", fontsize=14, fontweight="bold")
    plt.tight_layout()

    with img_out:
        clear_output(wait=True)
        plt.show()
    with ctrl_out:
        clear_output(wait=True)
        print("RESULTS\n")
        overall = mean([1 if d["correct"] else 0 for d in test_logs]) * 100
        print(f"Overall accuracy  : {overall:.1f}%")
        print(f"Avg study time    : {mean([d['duration_s'] for d in study_logs]):.2f}s")
        print(f"Avg test time     : {mean([d['duration_s'] for d in test_logs]):.2f}s")
        print(f"Total test trials : {len(test_logs)}")
        print(f"Total faces       : {len({d['path'] for d in data_log})}")

# ── launch ────────────────────────────────────────────────────
screen_start()